### Apresentação ✒️

Notebook destinado a elaboração de um estudo acerca da qualidade de uma mensagem. A sua importância se circunscreve a partir do momento no qual verifica-se que mensagens podem possuir diferentes qualidades, atreladas ao seu maior ou menor entendimento por parte de um interlocutor qualquer. Em termos matemáticos, isso pode ser concebido segundo o conceito da entropia - presente na teoria da informação - segundo o qual possibilita mensurar o nível de incerteza para um conjunto qualquer. 

Colocando para um contexto de linguagem natural - e conversacional - que pode ser desepenhado entre humano-humano e humano-modelo (no contexto da existência de modelos de linguagem generativa), isso é observado a partir do instante no qual uma mensagem pode gerar mais ou menos dúvida acerca do que foi dito. Para o primeiro cenário, isso poderia ser entendido pela possível presença, virtualmente infinito, de conteúdos que poderiam ser utilizados para responder a mensagem, enquanto para o segundo o oposto, havendo uma segmentação mediante à própria sentença que inicialmente já foi trazida pelo emissor. 

A partir da linguística, existem algumas perspectivas com as quais pode-se utilizar para conceber a qualidade de uma mensagem. Para o presente estudo, considerei a perspectiva morfossintática, na qual a partir da sintaxe da forma de uma sentença permite compreender a sua qualidade, se declarativa ou não declarativa. 

Mensagens declarativas são compreendidas a partir do momento em que possuem semântica, coesão e coerência bem elaboradas, ao passo que as não declarativas possuem o oposto, falhando em alguns desses eixos citados, fazendo com que possa haver perda de significado compreendido ou entendimento lógico necessário para o seu entendimento, prejudicando, portanto, a sua interpretação. 

O presente estudo buscou realizar essa identificação da qualidade de uma mensagem a partir de um formalismo matemático. Nesse sentido, será apresentado um coeficiente construído como a razão das classes gramaticais presentes numa sentença em relação às classes gramaticais que precisariam estar pelo menos uma vez para a construção de sentenças que fossem declarativas, ao mesmo tempo em que há a adaptação do modelo de entropia, que irá mensurar o nível de entropia em razão da frequência das classes gramaticias. Ao final, os valores proporcionados por tais modelos - além da quantidade de caracteres e as classes gramaticais *per se* - serviram como features para um modelo de machine learning clássico de classficação. 

Séra demonstrado estudos de correlação que fundamentam a utilização dos dois modelos iniciais citados, bem como métricas de classificação e matriz de confusão, que permite conceber a qualidade de predição e nível de concordância com a ground truth, respectivamente. Para compreender melhor o aspecto teórico para o qual esse notebook se refere, checar o seguinte artigo: A qualidade de uma resposta se relaciona com a de sua pergunta, meu caro Watson. 

#### Metodologia 🧪

O presente notebook está dividido a partir dos seguintes tópicos: 

1. Library
2. Funções utilizadas para o estudo (e os modelos)
3. Loading do dataset
4. Breve exploração acerca dos dados verificados 
5. Aplicação dos modelos considerados para a classificação
6. Featuring Engineering para o modelo de ML considerado
7. Resultados encontrados
8. Conclusão do estudo 

#### Library 🥀

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pickle

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import spacy

from pandas import DataFrame
from scipy.stats import entropy
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, cohen_kappa_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [3]:
nlp = spacy.load("pt_core_news_sm")

In [4]:
# Testando a tokenização encontrada pela
# biblioteca do Spacy

text = """\ 
Quero deixar queimar para não me apagar, 
Quero correr de novo e ser fogo 
Me espalhar para poder olhar o meu rosto 
"""

sentence = nlp(text)

tags = [token.pos_ for token in sentence]
tags

['NOUN',
 'SPACE',
 'VERB',
 'VERB',
 'VERB',
 'SCONJ',
 'ADV',
 'PRON',
 'VERB',
 'PUNCT',
 'SPACE',
 'VERB',
 'VERB',
 'ADP',
 'NOUN',
 'CCONJ',
 'AUX',
 'NOUN',
 'SPACE',
 'PRON',
 'VERB',
 'SCONJ',
 'VERB',
 'VERB',
 'DET',
 'DET',
 'NOUN',
 'SPACE']

#### Helper functions 🕸️

In [5]:
def calculate_coeficient(sentence: str) ->float:
    """ 
    """
    doc = nlp(sentence)

    gramma_class = {
        "NOUN", "VERB", "NUM", "AUX", 
        "ADP", "SYM", "PRO", "CCOJ", 
        "SCOJ", "ADJ", "DET", "ADV"
    }

    pos_tags = [token.pos_ for token in doc]

    numerator   = sum(1 for pos in pos_tags if pos in gramma_class)
    denominator = len(gramma_class)
    coeficient  = numerator/denominator
    
    return coeficient

In [154]:

def plot_confusion_matrix(
    y_true,
    y_pred,
    labels=None,
    normalize=False,
    title="Confusion Matrix"
):
    """
    Plota uma matriz de confusão usando Plotly.

    Parameters
    ----------
    y_true : array-like
        Valores reais.

    y_pred : array-like
        Valores preditos.

    labels : list, optional
        Classes na ordem desejada.

    normalize : bool, default=False
        Se True, exibe proporções em vez de contagens.

    title : str
        Título do gráfico.
    """

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        text_format = ".2%"
    else:
        text_format = "d"

    fig = px.imshow(
        cm,
        x=labels,
        y=labels,
        text_auto=text_format,
        aspect="auto",
        labels={
            "x": "Predicted",
            "y": "Actual",
            "color": "Count"
        },
        title=title
    )

    fig.update_layout(
        template="plotly_white"
    )

    fig.show()

In [ ]:
def count_grama_class(sentence: str) -> dict: 
    """
     Identifica a presença de classes gramaticais em uma sentença.

    Para cada classe gramatical monitorada, retorna 1 caso ela apareça
    ao menos uma vez na sentença e 0 caso contrário.

    Args:
        sentence (str): Texto a ser analisado.

    Returns:
        dict: Dicionário contendo as classes gramaticais como chaves
        e valores binários (0 ou 1) indicando sua presença. 
    """
    doc = nlp(sentence)

    gramma_class = {
        "NOUN", "VERB", "NUM", "AUX", 
        "ADP", "SYM", "PRO", "CCOJ", 
        "SCOJ", "ADJ", "DET", "ADV"
    }

    frequencys = {clas:0 for clas in gramma_class}

    for token in doc: 

        clas = token.pos_

        if clas in gramma_class: 
            
            frequencys[clas] = 1
    
    if all(frequency == 0 for frequency in frequencys.values()):    

        return {clas: 0 for clas in gramma_class}

    return frequencys

In [ ]:
def sum_grama_class(sentence: str) -> dict: 
    """
     Conta a frequência de cada classe gramatical em uma sentença.

    Percorre os tokens identificados pelo modelo NLP e contabiliza
    quantas vezes cada classe gramatical aparece.

    Args:
        sentence (str): Texto a ser analisado.

    Returns:
        dict: Dicionário contendo as classes gramaticais como chaves
        e suas respectivas frequências como valores. 
    """
    doc = nlp(sentence)

    gramma_class = {
        "NOUN", "VERB", "NUM", "AUX", 
        "ADP", "SYM", "PRO", "CCOJ", 
        "SCOJ", "ADJ", "DET", "ADV"
    }

    frequencys = {clas:0 for clas in gramma_class}

    for token in doc: 

        clas = token.pos_

        if clas in gramma_class: 
            
            frequencys[clas] += 1
    
    if all(frequency == 0 for frequency in frequencys.values()):    

        return {clas: 0 for clas in gramma_class}

    return frequencys

In [ ]:
def calculate_entropy(gramma_clas_dictionary: dict) -> float: 
    """  
    Calcula a entropia de Shannon a partir das frequências gramaticais.

    As frequências são convertidas em probabilidades e utilizadas
    para calcular a entropia utilizando logaritmo natural.

    Args:
        gramma_clas_dictionary (dict): Dicionário contendo frequências
        de classes gramaticais.

    Returns:
        float: Valor da entropia calculada.
    """
    log_base = np.e 

    array_grama = np.array(
        list(
            gramma_clas_dictionary.values(), 
        ), 
        dtype=float
    )

    probs         = (array_grama / array_grama.sum())
    entropy_value = entropy(probs, base=log_base) 

    return float(entropy_value)

In [ ]:
def correlation_table(
        dataset: DataFrame, 
        column_name_1 : str, 
        column_name_2 : str, 
        ) -> DataFrame: 
        """  
        Gera uma tabela de contingência entre duas colunas categóricas.

        Utiliza o método `pandas.crosstab` para calcular as frequências
        conjuntas entre os valores das colunas informadas.

        Args:
                dataset (DataFrame): Conjunto de dados de entrada.
                column_name_1 (str): Nome da primeira coluna.
                column_name_2 (str): Nome da segunda coluna.

        Returns:
                DataFrame: Tabela de contingência contendo as frequências
                observadas para cada combinação de categorias.
        """
        crosstab = pd.crosstab(
                dataset[column_name_1], 
                dataset[column_name_2]
        )

        return crosstab

In [ ]:
def discretize_continuos_values(
        dataset: DataFrame, 
        column_name: str, 
        id_column_name : str = "id", 
        threshold: float = 0.4
        ):  
        """  
        Discretiza uma variável contínua utilizando um limiar fixo.

        Valores maiores ou iguais ao limiar recebem 1. Valores abaixo
        do limiar recebem 0.

        Args:
                dataset (DataFrame): Conjunto de dados de entrada.
                column_name (str): Coluna contendo os valores contínuos.
                id_column_name (str, optional): Coluna identificadora que será
                preservada no resultado. Padrão: "id".
                threshold (float, optional): Limiar utilizado para a
                discretização. Padrão: 0.4.

        Returns:
                DataFrame: DataFrame contendo o identificador e a coluna
                `discretize_value` com os valores binarizados.
        """
        base = []

        for idx, row in dataset.iterrows(): 
                
                if row[column_name] >= threshold: 
                       discretize_value = 1 

                else: 
                       discretize_value = 0
        
                base.append(
                        {
                                id_column_name : row[id_column_name], 
                                "discretize_value": discretize_value
                        }
                )
        
        base = DataFrame(base)
        return base

#### Carregando o dataset

In [11]:
df = pd.read_excel("../data/dataset_mensagens_nlg.xlsx")

In [12]:
df.head(2)

,id,mensagem,tema,rotulo
0,1,Narrativas urbanas violentas geralmente explor...,Tokyo Ghoul,OK
1,2,Dor sob pele estranha?,Tokyo Ghoul,NOK


In [13]:
df.shape

(200, 4)

In [14]:
df["rotulo"].value_counts()

rotulo
NOK    102
OK      98
Name: count, dtype: int64

In [15]:
df["tema"].value_counts()

tema
Literatura Russa    72
Tokyo Ghoul         66
City Pop            62
Name: count, dtype: int64

Checando a qualidade dos dados

In [16]:
df.isnull().sum()

id          0
mensagem    0
tema        0
rotulo      0
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(0)

#### Entendendo a morfologia das sentenças a partir do coeficiente de densidade gramatical e do nível de entropia 

- Quantidade de caracteres

In [18]:
lengh_char_list = []

for i in tqdm(range(df.shape[0]), "Medindo a qt. de caracteres"): 

    lenght_char = len(df["mensagem"].iloc[i])
    lengh_char_list.append(lenght_char)

df["lenght_char"] = lengh_char_list

Medindo a qt. de caracteres: 100%|██████████| 200/200 [00:00<00:00, 73033.33it/s]


In [19]:
df.head(2)

,id,mensagem,tema,rotulo,lenght_char
0,1,Narrativas urbanas violentas geralmente explor...,Tokyo Ghoul,OK,72
1,2,Dor sob pele estranha?,Tokyo Ghoul,NOK,22


In [20]:
bins   = [0, 40, 80, 120, 160, 500]
labels = ["0 - 40","40 - 80","80 - 120","120 - 160","160 - 500"]

In [21]:
df["interval_char"] = pd.cut(
    df["lenght_char"],
    bins   = bins, 
    labels = labels
)

In [22]:
correlation_table(
    dataset       = df, 
    column_name_1 = "rotulo", 
    column_name_2 = "interval_char"
)

interval_char,0 - 40,40 - 80,80 - 120
rotulo,,,
NOK,102,0,0
OK,0,86,12


- Coeficiente

In [23]:
coeficient_list = []

for i in tqdm(range(df.shape[0]), desc="Calculando o coeficiente de densidade gramatical"): 

    sentence            = df["mensagem"].iloc[i]
    coeficient_sentence = calculate_coeficient(sentence=sentence)

    coeficient_list.append(coeficient_sentence)

Calculando o coeficiente de densidade gramatical: 100%|██████████| 200/200 [00:01<00:00, 155.19it/s]


In [24]:
df["coeficient_gramma"] = coeficient_list

In [25]:
bins   = [0, 0.4, 0.6, 0.8, 1.2, 1.6, 2]
labels = ["0 - 0.4", "0.4 - 0.6","0.6 - 0.8","0.8 - 1.2","1.2 - 1.6","1.6 - 2.0"]

In [26]:
df["interval_coeficient"] = pd.cut(
    df["coeficient_gramma"],
    bins    = bins, 
    labels  = labels
)

In [27]:
correlation_table(
    dataset       = df, 
    column_name_1 = "rotulo", 
    column_name_2 = "interval_coeficient"
)

interval_coeficient,0 - 0.4,0.4 - 0.6,0.6 - 0.8,0.8 - 1.2
rotulo,,,,
NOK,99,3,0,0
OK,0,22,39,37


- Entropia

In [28]:
gramma_clas_list = [] 

for i in tqdm(range(df.shape[0]), desc="Contando a apariação das classes gramaticais"): 

    sentence = df["mensagem"].iloc[i]
    gramma_class = count_grama_class(sentence=sentence)

    gramma_clas_list.append(gramma_class)

Contando a apariação das classes gramaticais: 100%|██████████| 200/200 [00:01<00:00, 151.49it/s]


In [29]:
df["count_gramma_clas"] = gramma_clas_list

In [39]:
entropy_gramma_list = [] 

for i in tqdm(range(df.shape[0]), desc="Calculando o nível de entropia gramatical"): 

    gramma_array  = df["count_gramma_clas"].iloc[i]
    entropy_value = calculate_entropy(gramma_clas_dictionary=gramma_array)

    entropy_gramma_list.append(entropy_value)

Calculando o nível de entropia gramatical: 100%|██████████| 200/200 [00:00<00:00, 3697.88it/s]


In [41]:
df["entropy_gramma"] = entropy_gramma_list

In [42]:
correlation_table(
    dataset       = df, 
    column_name_1 = "rotulo", 
    column_name_2 = "entropy_gramma"
)

entropy_gramma,0.000000,0.693147,1.098612,1.386294,1.609438,1.791759,1.945910,1.945910
rotulo,,,,,,,,
NOK,23,61,13,5,0,0,0,0
OK,0,0,12,17,30,33,3,3


Analisando cada um dos marcadores contemplados, verifica-se que parece existir uma correlação positiva entre maior quantidade de caracteres e valores para o coeficiente de desnsidade gramatical, bem como o de entropia em relação às mensagens tidas como declarativas (OK). Como mostrado no artigo citado na apresentação desse notebook, isso ocorre em decorrência de uma certa colinearidade entre mensagens melhores ecritas com maior incidência de termos - atrelados a determinadas classes gramaticais - para a construção da sentença, permitindo-a que seja mais coerente, coesa e semântica. Com a adoção de mais termos, tenciona a existir também uma maior quantidade de caracteres para tanto.

Em virtude disso, poderia-se supor que, a partir da correlação verificada, pode-se clivar uma mensagem OK de uma NOK em razão de um determinado limiar encontrado para cada um dos modelos concebidos. Por exemplo, o coeficiente poderia ter um limiar de 0.4, enquanto que o de entropia de 1. 

#### Adicionando um modelo de ML

- Featuring Engineering

In [135]:
new_df = df.copy()
new_df.head(2)

,id,mensagem,tema,rotulo,lenght_char,interval_char,coeficient_gramma,interval_coeficient,count_gramma_clas,entropy_gramma
0,1,Narrativas urbanas violentas geralmente explor...,Tokyo Ghoul,OK,72,40 - 80,0.583333,0.4 - 0.6,"{'NUM': 0, 'SYM': 0, 'VERB': 1, 'NOUN': 1, 'SC...",1.386294
1,2,Dor sob pele estranha?,Tokyo Ghoul,NOK,22,0 - 40,0.333333,0 - 0.4,"{'NUM': 0, 'SYM': 0, 'VERB': 0, 'NOUN': 1, 'SC...",1.098612


In [136]:
new_df = new_df[
    [
        "id", "mensagem", "lenght_char", "coeficient_gramma", 
        "entropy_gramma", "count_gramma_clas", "rotulo", 
    ]
]

Criando um novo dataset considerando a ocorrência das classes gramaticais, formando um vetor esparso

In [45]:
new_df["count_gramma_clas"][0]

{'NUM': 0,
 'SYM': 0,
 'VERB': 1,
 'NOUN': 1,
 'SCOJ': 0,
 'ADJ': 1,
 'ADV': 1,
 'ADP': 0,
 'AUX': 0,
 'DET': 0,
 'CCOJ': 0,
 'PRO': 0}

In [46]:
num_list  = [] 
sym_list  = []
verb_list = []
noun_list = []
scoj_list = []
adj_list  = []
adv_list  = []
adp_list  = []
aux_list  = []
det_list  = []
ccoj_list = []
pro_list  = []

In [48]:
for i in tqdm(range(new_df.shape[0]), desc="Calculando o nível de entropia gramatical"): 

    freq_class  = new_df["count_gramma_clas"].iloc[i]
    
    num  = freq_class["NUM"]
    sym  = freq_class["SYM"]
    verb = freq_class["VERB"]
    noun = freq_class["NOUN"]
    scoj = freq_class["SCOJ"]
    adj  = freq_class["ADJ"]
    adv  = freq_class["ADV"]
    adp  = freq_class["ADP"]
    aux  = freq_class["AUX"]
    det  = freq_class["DET"]
    ccoj = freq_class["CCOJ"]
    pro  = freq_class["PRO"]

    num_list.append(num)
    sym_list.append(sym)
    verb_list.append(verb)
    noun_list.append(noun)
    scoj_list.append(scoj)
    adj_list.append(adj)
    adv_list.append(adv)
    adp_list.append(adp)
    aux_list.append(aux)
    det_list.append(det)
    ccoj_list.append(ccoj)
    pro_list.append(pro)

Calculando o nível de entropia gramatical: 100%|██████████| 200/200 [00:00<00:00, 12170.63it/s]


In [49]:
df_for_ml_model = DataFrame(
    { 
        "adjetivo"           : adj_list, 
        "preposicao"         : adp_list,  
        "adverbio"           : adv_list, 
        "verbo auxiliar"     : aux_list, 
        "substantivo"        : noun_list, 
        "verbo"              : verb_list, 
        "numeral"            : num_list, 
        "simbolo"            : sym_list, 
        "c. subordinativa"   : scoj_list, 
        "c. coordenativa"    : ccoj_list, 
        "adjuntos adnomiais" : det_list
    }
)

In [51]:
df_for_ml_model["lengh_char"]        = new_df["lenght_char"]
df_for_ml_model["gramma_coeficient"] = new_df["coeficient_gramma"]
df_for_ml_model["entropy_gramma"]    = new_df["entropy_gramma"]
df_for_ml_model["label"]             = new_df["rotulo"]

In [52]:
df_for_ml_model.head(2)

,adjetivo,preposicao,adverbio,verbo auxiliar,substantivo,verbo,numeral,simbolo,c. subordinativa,c. coordenativa,adjuntos adnomiais,lengh_char,gramma_coeficient,entropy_gramma,label
0,1,0,1,0,1,1,0,0,0,0,0,72,0.583333,1.386294,OK
1,1,1,0,0,1,0,0,0,0,0,0,22,0.333333,1.098612,NOK


In [53]:
df_for_ml_model.shape

(200, 15)

Ajustando a `label`que fique num formato inteiro-booleano

In [ ]:
df_for_ml_model["label"] = df_for_ml_model["label"].map(
    {
        "OK" : 1, 
        "NOK": 0
    }
)

In [55]:
df_for_ml_model.head(2)

,adjetivo,preposicao,adverbio,verbo auxiliar,substantivo,verbo,numeral,simbolo,c. subordinativa,c. coordenativa,adjuntos adnomiais,lengh_char,gramma_coeficient,entropy_gramma,label
0,1,0,1,0,1,1,0,0,0,0,0,72,0.583333,1.386294,1
1,1,1,0,0,1,0,0,0,0,0,0,22,0.333333,1.098612,0


##### Modelos de ML considerados

Os modelos de ML considerados para a classificação da qualidade da mensagem serão : 

- Random Forest 
- Regressão logística (aka master Tete's model hahah)
- SVM

Durante a elaboração do modelo percebi que ambos tiveram bom ajuste aos dados não exigindo otimização a partir de alguns frameworks, como o Optuna. 

In [75]:
SEED = 22

In [76]:
X = df_for_ml_model.drop(columns="label")

In [77]:
y = df_for_ml_model["label"]

In [98]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    stratify     = y, 
    shuffle      = True,
    test_size    = 0.3, 
    random_state = SEED
)

- Random Forest (baseline)

In [100]:
random_forest_baseline = RandomForestClassifier(
    random_state = SEED
)

In [101]:
random_forest_baseline.fit(X_train, y_train)

y_pred_rf = random_forest_baseline.predict(X_test)

In [102]:
class_report = classification_report(
    y_test, y_pred_rf
)

print(class_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        31
           1       1.00      1.00      1.00        29

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



In [123]:
random_forest = RandomForestClassifier(
    n_estimators = 100, 
    max_depth    = 10, 
    random_state = SEED
)

In [124]:
random_forest.fit(X_train, y_train)

y_pred_rf = random_forest.predict(X_test)

In [128]:
class_report = classification_report(
    y_test, y_pred_rf
)

print(class_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        31
           1       1.00      1.00      1.00        29

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



In [126]:
feature_importances = random_forest.feature_importances_
std = np.std([tree.feature_importances_ for tree in random_forest.estimators_], axis=0)

In [ ]:
forest_importances = pd.Series(
    feature_importances,
    index=X.columns
).sort_values()

fig = go.Figure(
    go.Bar(
        x = forest_importances.values,
        y = forest_importances.index,
        orientation ="h",
        error_x     =dict(
            type    = "data",
            array   = std[forest_importances.index.argsort()],
            visible = True
        )
    )
)

fig.update_layout(
    title       = "Feature Importances (MDI)",
    xaxis_title = "Mean decrease in impurity",
    yaxis_title = "Feature",
    template    = "plotly_white",
    height      = max(500, len(X.columns) * 25)
)

fig.show()

Considerando o gráfico das features mais importantes para a classificação do modelo de regressão logística, pode-se conceber que como importantes preditores acerca da qualidade de uma mensagem estão os coeficientes encontrados pello modelo de densidade gramatical e nível de entropia, além da quantidade de caracteres, sendo seguida por algumas outras classes gramaticais, como adjetivo por exemplo. Num cenário de redução de dimensionalidade para diminuição de overfitting, poderia manter somente as features que foram vistas como mais importantes em detrimento das demais. 

- Regressão logística (baseline)

In [ ]:
log_reg_baseline = LogisticRegression(random_state=SEED)

log_reg_baseline.fit(X_train, y_train)

y_pred_rg_log = log_reg_baseline.predict(X_test)

In [104]:
class_report = classification_report(
    y_test, y_pred_rg_log
)

print(class_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        31
           1       1.00      1.00      1.00        29

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



In [129]:
log_reg = LogisticRegression(
    random_state = SEED, 
    multi_class  = "ovr", 
    solver       = "lbfgs", 
    class_weight = "balanced"
)

log_reg.fit(X_train, y_train)

y_pred_rg_log = log_reg.predict(X_test)

In [130]:
class_report = classification_report(
    y_test, y_pred_rg_log
)

print(class_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        31
           1       1.00      1.00      1.00        29

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



- SVM (baseline)

In [106]:
svm_model = svm.SVC()

In [107]:
svm_model.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [108]:
y_svm_predict = svm_model.predict(X_test)

In [109]:
class_report = classification_report(
    y_test, y_svm_predict
)

print(class_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        31
           1       1.00      1.00      1.00        29

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



##### Selecionando o modelo

Como para o presente cenário todos os modelos convergiram na melhor predição possível, irei selecionar dentre esses aquele que normalmente tende a ser mais consistente em inúmeros cenários de ML: regressão logística. 

In [131]:
log_reg = LogisticRegression(
    random_state = SEED, 
    multi_class  = "ovr", 
    solver       = "lbfgs", 
    class_weight = "balanced"
)

log_reg.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,22
,solver,'lbfgs'
,max_iter,100
,multi_class,'ovr'


In [133]:
# Salvando o modelo de ML selecionado para utilização posterior. 

with open('log_reg_model.pkl', 'wb') as file:
    pickle.dump(log_reg, file)

#### Resultados encontrados

Discretizando os valores encontrados pelo modelo baseado na densidade gramatical e pelo nível de entropia.

In [137]:
new_df.head(1)

,id,mensagem,lenght_char,coeficient_gramma,entropy_gramma,count_gramma_clas,rotulo
0,1,Narrativas urbanas violentas geralmente explor...,72,0.583333,1.386294,"{'NUM': 0, 'SYM': 0, 'VERB': 1, 'NOUN': 1, 'SC...",OK


In [141]:
dicretize_coeficient = discretize_continuos_values(
    dataset     = new_df, 
    column_name = "coeficient_gramma",  
)

new_df["disc_coeficient"] = dicretize_coeficient["discretize_value"]

In [142]:
dicretize_coeficient = discretize_continuos_values(
    dataset     = new_df, 
    column_name = "entropy_gramma",  
)

new_df["disc_entropy"] = dicretize_coeficient["discretize_value"]

In [143]:
new_df["rotulo"] = new_df["rotulo"].map(
    {
        "OK"  : 1, 
        "NOK" : 0
    }
)

- Coeficiente

In [148]:
clas_report = classification_report(
    new_df["rotulo"], 
    new_df["disc_coeficient"]
)

print(clas_report)

              precision    recall  f1-score   support

           0       1.00      0.97      0.99       102
           1       0.97      1.00      0.98        98

    accuracy                           0.98       200
   macro avg       0.99      0.99      0.98       200
weighted avg       0.99      0.98      0.99       200



In [ ]:
plot_confusion_matrix(
   new_df["rotulo"], 
   new_df["disc_coeficient"] 
)

- Entropia

In [149]:
clas_report = classification_report(
    new_df["rotulo"], 
    new_df["disc_entropy"]
)

print(clas_report)

              precision    recall  f1-score   support

           0       1.00      0.23      0.37       102
           1       0.55      1.00      0.71        98

    accuracy                           0.60       200
   macro avg       0.78      0.61      0.54       200
weighted avg       0.78      0.60      0.54       200



In [ ]:
plot_confusion_matrix(
   new_df["rotulo"], 
   new_df["disc_entropy"] 
)

- ML Model

In [150]:
y_pred_ml_model = log_reg.predict(X)

In [152]:
clas_report = classification_report(
    df_for_ml_model["label"], 
    y_pred_ml_model
)

print(clas_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       102
           1       1.00      1.00      1.00        98

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [ ]:
plot_confusion_matrix(
   df_for_ml_model["label"], 
   y_pred_ml_model
)

#### Conclusão

Analisando os resultados e validação anterior acerca da fundamentação sobre a possibilidade de usar os modelos considerados para a classificação da qualidade das mensagens declarativas e não declarativas, notou-se êxito. Um ponto de atenção é na qualidade de 100% que o modelo de ML apresenta, podendo se direcionar a um overfitting presente em seu treinamento. Contudo, mesmo para os modelos de coeficiente e de entropia, observou-se boa qualidade de predição, ainda que o segundo tenha falhado na classificação de mensagens NOK, evidenciando as fragilidades nas quais esses modelos podem possuir em razão da sua perspectiva tão somente morfossintática de análise para a classificação das mensagens. 